# CS 7643 Assignment 1: Implement and Train a Network on MNIST

In this assignment you will build a complete neural network training pipeline from scratch to classify [MNIST handwritten digits](https://yann.lecun.com/exdb/mnist/). You will implement:

1. **Data loading & batching** — split data into train/val and organize into mini-batches
2. **Activation & loss functions** — ReLU, Sigmoid, Softmax, Cross-Entropy
3. **Two models** — Softmax Regression and a Two-Layer MLP
4. **SGD optimizer** with L2 regularization
5. **Hyperparameter experiments** — learning rate, regularization, and tuning for best accuracy

Be sure to use the report template provided and fill in your information on the first page.

---
## 0. Environment Setup

Run the cells below to configure your environment. This works on both Google Colab and local machines (VS Code, JupyterLab, etc.) without any manual changes when running locally.

> **Tip:** The [autoreload](https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html) extension lets you edit `.py` files and have changes reflected here immediately — no kernel restart needed.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Install dependencies into the current kernel (runs quietly if already installed)
import subprocess, sys
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'numpy', 'matplotlib', 'requests', 'tqdm', 'pytest']
)

In [ ]:
import sys
import os
import numpy as np
import math

# --- Detect environment and set PROJECT_ROOT ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO (Colab only): set this to your folder path inside My Drive
    # Example: 'Fall2024/A1'
    GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = None  # <-- FILL THIS IN
    PROJECT_ROOT = os.path.join('drive', 'MyDrive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
else:
    # Running locally — automatically resolves to the notebook's directory
    PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))

# Add project root to Python path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Convenience path for tests
TEST_DIR = os.path.join(PROJECT_ROOT, 'tests')

# Keep backward compat with helper scripts that reference this variable
GOOGLE_DRIVE_PATH = PROJECT_ROOT

print(f'Environment: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Project root: {PROJECT_ROOT}')

### Sanity Check

The cell below verifies that all source files are importable and shows their last-modified timestamps. You should see `Roger that from ...` messages for each file.

In [ ]:
import time
os.environ["TZ"] = "US/Eastern"
try:
    time.tzset()
except AttributeError:
    pass  # Windows does not support tzset

from cs7643.env_prob import say_hello_do_you_copy
say_hello_do_you_copy(PROJECT_ROOT)

---
## 1. Load the MNIST Dataset

We use a CSV-based version of MNIST (60 000 training images, 10 000 test images). Each image is 28×28 = 784 pixels, normalized to [0, 1]. The first run will download the data (~100 MB).

In [ ]:
from cs7643.mnist import MNIST

train_ds = MNIST(os.path.join(PROJECT_ROOT, 'data'), download=True, train=True)
test_ds = MNIST(os.path.join(PROJECT_ROOT, 'data'), download=True, train=False)

print(f'Training set : {len(train_ds)} samples')
print(f'Test set     : {len(test_ds)} samples')

### Visualize a few samples

Always look at your data before training. This confirms the download worked and gives intuition about the task.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img = np.array(train_ds.data[i]).reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {train_ds.targets[i]}')
    ax.axis('off')
plt.suptitle('Sample MNIST Images', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Data Preparation

Before training we need to:
1. **Split** the 60k training set into 80% train / 20% validation (first 48k for training, last 12k for validation)
2. **Batch** the data into mini-batches for efficient gradient computation

You will implement both operations in `./data/data_processing.py`.

### 2a. Train/Validation Split

Implement `load_mnist_trainval` in `./data/data_processing.py`.

**Expected output:**
- `train_data`: 48 000 samples
- `val_data`: 12 000 samples

In [ ]:
from data.data_processing import load_mnist_trainval, generate_batched_data

train_data, train_label, val_data, val_label = load_mnist_trainval(
    train_ds.train_data, train_ds.train_labels
)

print(f'train_data samples : {len(train_data)}  (expected: 48000)')
print(f'val_data samples   : {len(val_data)}  (expected: 12000)')

In [ ]:
!pytest {os.path.join(TEST_DIR, 'test_loading.py') + '::TestLoading::test_load_mnist'} -v

### 2b. Batching

Implement `generate_batched_data` in `./data/data_processing.py`.

**Expected output:**
- Number of batches: `ceil(60000 / 128)` = 469
- Samples per batch: 128 (last batch may be smaller)

In [ ]:
batched_train_data, batched_train_label = generate_batched_data(
    train_ds.train_data, train_ds.train_labels,
    batch_size=128, shuffle=True, seed=1024
)

print(f'Number of batches     : {len(batched_train_data)}  (expected: 469)')
print(f'Samples in first batch: {len(batched_train_data[0])}  (expected: 128)')

In [ ]:
!pytest {os.path.join(TEST_DIR, 'test_loading.py') + '::TestLoading::test_generate_batch'} -v

---
## 3. Model Implementation

You will implement two networks from scratch. Both models store weights in a `weights` dictionary and gradients in a `gradients` dictionary. The `forward` method computes loss, accuracy, and (during training) gradients.

### Shared Utility Functions

First, implement these helper functions in `./models/_base_network.py`:

| Function | Description |
|----------|-------------|
| `sigmoid(x)` | Element-wise sigmoid activation |
| `sigmoid_dev(x)` | Derivative of sigmoid |
| `ReLU(x)` | Element-wise ReLU activation |
| `ReLU_dev(x)` | Derivative of ReLU |
| `softmax(x)` | Row-wise softmax (for numerical stability, subtract max) |
| `cross_entropy_loss(...)` | Compute CE loss from softmax outputs |
| `compute_accuracy(...)` | Classification accuracy for a batch |

### 3a. Activation Functions

Implement `sigmoid`, `sigmoid_dev`, `ReLU`, `ReLU_dev`, and `softmax` in `./models/_base_network.py`.

**Reminder:**
- $\text{sigmoid}(x) = \frac{1}{1 + e^{-x}}$, derivative: $\sigma(x)(1 - \sigma(x))$
- $\text{ReLU}(x) = \max(0, x)$, derivative: $1$ if $x > 0$, else $0$

In [ ]:
# Run all activation tests at once
!pytest {os.path.join(TEST_DIR, 'test_activation.py')} -v

### 3b. Loss Function & Accuracy

Implement `cross_entropy_loss` and `compute_accuracy` in `./models/_base_network.py`.

**Cross-Entropy Loss:**
$$L_{CE} = -\frac{1}{N} \sum_{i=1}^N \log(p_{i, y_i})$$

where $p_{i, y_i}$ is the softmax probability assigned to the correct class for sample $i$.

In [ ]:
!pytest {os.path.join(TEST_DIR, 'test_loss.py')} -v

### 3c. Network Forward & Backward Passes

Now implement the `forward` method in both model files. Here's the architecture comparison:

| | Softmax Regression (`softmax_regression.py`) | Two-Layer MLP (`two_layer_nn.py`) |
|---|---|---|
| **Layers** | Input → FC → ReLU → Softmax | Input → FC₁ → Sigmoid → FC₂ → Softmax |
| **Bias** | No bias | Yes (b1, b2) |
| **Weights** | `W` (input_size × num_classes) | `W1`, `b1`, `W2`, `b2` |
| **File** | `./models/softmax_regression.py` | `./models/two_layer_nn.py` |

**Key implementation notes:**
- Apply softmax before computing cross-entropy loss
- When `mode='train'`, compute and store gradients in `self.gradients`
- When `mode='test'`, just return `(loss, accuracy)`

> **Debugging tip:** If your gradient check fails, verify the forward pass produces the correct loss first. Common issues: forgetting to average over the batch, or applying ReLU in the wrong place.

In [ ]:
# Test Softmax Regression
!pytest {os.path.join(TEST_DIR, 'test_network.py') + '::TestNetwork::test_one_layer_softmax_relu'} -v

In [ ]:
# Test Two-Layer MLP
!pytest {os.path.join(TEST_DIR, 'test_network.py') + '::TestNetwork::test_two_layer_net'} -v

---
## 4. Optimizer

Implement SGD with L2 regularization in `./optimizer/`.

**Overall loss with regularization:**
$$J = L_{CE} + \frac{1}{2}\lambda \sum_{i=1}^N w_i^2$$

**SGD update rule:**
$$\theta^{t+1} = \theta^{t} - \eta \, \nabla_{\theta} J(\theta)$$

**Your tasks:**
1. Implement `apply_regularization` in `./optimizer/_base_optimizer.py`
   - Add $\lambda \cdot w$ to the gradient for weight matrices
   - Do **NOT** regularize bias terms
2. Implement `update` in `./optimizer/sgd.py`
   - Standard gradient descent step: $w \leftarrow w - \eta \cdot \nabla w$

In [ ]:
# Test regularization and SGD
!pytest {os.path.join(TEST_DIR, 'test_training.py') + '::TestTraining::test_regularization'} -v
!pytest {os.path.join(TEST_DIR, 'test_training.py') + '::TestTraining::test_sgd'} -v

In [ ]:
# End-to-end training tests (combines model + optimizer)
!pytest {os.path.join(TEST_DIR, 'test_training.py') + '::TestTraining::test_one_layer_train'} -v
!pytest {os.path.join(TEST_DIR, 'test_training.py') + '::TestTraining::test_two_layer_train'} -v

---
## Run All Tests

Use this cell as a quick pass/fail check before moving to experiments.

In [ ]:
!pytest {TEST_DIR} -v --tb=short

---
## 5. Experiments

Now that your pipeline works, you will tune hyperparameters on the **two-layer MLP** and report observations.

Edit `./configs/config_exp.json` to change parameters. The default config is:

```json
{
  "model": "TwoLayerNet",
  "learning_rate": 0.1,
  "reg": 0.001,
  "batch_size": 64,
  "epochs": 10,
  "hidden_size": 128
}
```

### What to try

| Hyperparameter | Suggested Range | Notes |
|---|---|---|
| `learning_rate` | 1e-4 to 1.0 | Try 0.001, 0.01, 0.1, 0.5 |
| `reg` (L2) | 0 to 0.1 | Try 0, 0.001, 0.01, 0.1 |
| `hidden_size` | 32 to 512 | Larger = more capacity but slower |
| `batch_size` | 32 to 512 | Smaller = noisier gradients, more updates per epoch |

### Requirements
- Try different learning rates and report observations
- Try different regularization coefficients and report observations
- Tune at least **3 hyperparameters** (not counting epochs) for best accuracy
- Best model should show **marked improvement** over defaults
- When tuning one parameter, keep others at their default values

> **Tip:** Change `config_file` below to select which config to run. For local execution, just edit the string directly.

In [ ]:
from cs7643.solver import run_experiment, plot_curve

# Change this to run different configs: "config_exp", "config_softmax", "config_two_layer_net"
config_file = "config_exp"  # @param ["config_exp", "config_softmax", "config_two_layer_net"]

train_loss_history, train_acc_history, valid_loss_history, valid_acc_history = run_experiment(config_file)

# Plot accuracy curves
plot_curve(train_loss_history, train_acc_history, valid_loss_history, valid_acc_history)

# Plot loss curves
plot_curve(train_loss_history, train_acc_history, valid_loss_history, valid_acc_history, plot_type='loss')

---
## 6. Submit Your Work

Run the cell below to create a `.zip` file for Gradescope submission.

**Important:** Save all `.py` files before running this cell.

In [ ]:
from cs7643.submit import make_a1_submission

make_a1_submission(PROJECT_ROOT)